# MindCheck Model Testing
## Testing Depression Detection Model with Various Scenarios

This notebook tests the model to verify it's not overfitting on specific features like `suicidal_thoughts`.

### Setup: Load Model and Scaler

In [ ]:
import pickle
import numpy as np
import pandas as pd
from google.colab import files

print("Upload model.pkl and scaler.pkl files")
uploaded = files.upload()

# Load model and scaler
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print("\n✅ Model and scaler loaded successfully!")
print(f"Model type: {type(model).__name__}")

### Helper Functions (Same as Flask API)

In [ ]:
def map_sleep_duration(sleep_str):
    """Map sleep duration string to numeric value"""
    mapping = {
        "Kurang dari 5 jam": 4.0,
        "5-6 jam": 5.5,
        "7-8 jam": 7.5,
        "Lebih dari 8 jam": 9.0
    }
    return mapping.get(sleep_str, 7.5)

def map_dietary_habits(diet_str):
    """Map dietary habits string to numeric value"""
    mapping = {
        "Tidak Sehat": 0,
        "Cukup": 1,
        "Sehat": 2
    }
    return mapping.get(diet_str, 1)

def map_binary(value_str, positive_value):
    """Map binary string to numeric (0 or 1)"""
    return 1 if value_str == positive_value else 0

def predict_depression(data):
    """Make prediction with the same logic as Flask API"""
    
    # Map string values to numeric
    sleep_numeric = map_sleep_duration(data['sleep_duration'])
    diet_numeric = map_dietary_habits(data['dietary_habits'])
    
    # Map binary values
    gender_numeric = map_binary(data['gender'], "Laki-laki")
    suicidal_numeric = map_binary(data['suicidal_thoughts'], "Pernah")
    family_history_numeric = map_binary(data['family_history'], "Ya")
    
    # Create feature array
    features = np.array([[
        gender_numeric,
        float(data['age']),
        float(data['academic_pressure']),
        float(data['study_satisfaction']),
        sleep_numeric,
        diet_numeric,
        suicidal_numeric,
        float(data['study_hours']),
        float(data['financial_stress']),
        family_history_numeric
    ]])
    
    # Scale features
    features_scaled = scaler.transform(features)
    
    # Make prediction
    prediction = int(model.predict(features_scaled)[0])
    
    # Get confidence
    try:
        probabilities = model.predict_proba(features_scaled)[0]
        confidence = float(probabilities[prediction])
    except:
        confidence = 0.75
    
    return {
        'prediction': prediction,
        'prediction_label': "No Depression" if prediction == 0 else "Depression",
        'confidence': round(confidence, 3),
        'features': features[0],
        'features_scaled': features_scaled[0]
    }

print("✅ Helper functions defined")

### Test Case 1: Very Low Risk Profile
**Student with excellent mental health indicators**

In [ ]:
test_low_risk = {
    "gender": "Perempuan",
    "age": 20,
    "academic_pressure": 2,
    "study_satisfaction": 4,
    "sleep_duration": "7-8 jam",
    "dietary_habits": "Sehat",
    "suicidal_thoughts": "Tidak Pernah",
    "study_hours": 6,
    "financial_stress": 2,
    "family_history": "Tidak"
}

result = predict_depression(test_low_risk)
print("=" * 60)
print("TEST 1: VERY LOW RISK PROFILE")
print("=" * 60)
print(f"Prediction: {result['prediction_label']}")
print(f"Confidence: {result['confidence']}")
print(f"\nInput Features:")
for key, value in test_low_risk.items():
    print(f"  {key}: {value}")
print(f"\nNumeric Features: {result['features']}")

### Test Case 2: Moderate Risk (No Suicidal Thoughts)
**Testing if model detects depression WITHOUT suicidal thoughts**

In [ ]:
test_moderate_no_suicidal = {
    "gender": "Laki-laki",
    "age": 22,
    "academic_pressure": 5,
    "study_satisfaction": 2,
    "sleep_duration": "Kurang dari 5 jam",
    "dietary_habits": "Tidak Sehat",
    "suicidal_thoughts": "Tidak Pernah",  # No suicidal thoughts
    "study_hours": 10,
    "financial_stress": 5,
    "family_history": "Ya"
}

result = predict_depression(test_moderate_no_suicidal)
print("=" * 60)
print("TEST 2: HIGH RISK BUT NO SUICIDAL THOUGHTS")
print("=" * 60)
print(f"Prediction: {result['prediction_label']}")
print(f"Confidence: {result['confidence']}")
print(f"\nInput Features:")
for key, value in test_moderate_no_suicidal.items():
    print(f"  {key}: {value}")
print(f"\nNumeric Features: {result['features']}")

### Test Case 3: Low Risk with Suicidal Thoughts
**Testing if suicidal thoughts alone triggers depression prediction**

In [ ]:
test_low_risk_with_suicidal = {
    "gender": "Perempuan",
    "age": 19,
    "academic_pressure": 2,
    "study_satisfaction": 4,
    "sleep_duration": "7-8 jam",
    "dietary_habits": "Sehat",
    "suicidal_thoughts": "Pernah",  # Has suicidal thoughts
    "study_hours": 5,
    "financial_stress": 1,
    "family_history": "Tidak"
}

result = predict_depression(test_low_risk_with_suicidal)
print("=" * 60)
print("TEST 3: LOW RISK BUT HAS SUICIDAL THOUGHTS")
print("=" * 60)
print(f"Prediction: {result['prediction_label']}")
print(f"Confidence: {result['confidence']}")
print(f"\nInput Features:")
for key, value in test_low_risk_with_suicidal.items():
    print(f"  {key}: {value}")
print(f"\nNumeric Features: {result['features']}")

### Test Case 4: Your Previous Test (High Risk + Suicidal)
**Reproducing the test case from mobile app logs**

In [ ]:
test_previous = {
    "gender": "Laki-laki",
    "age": 21,
    "academic_pressure": 4,
    "study_satisfaction": 3,
    "sleep_duration": "5-6 jam",
    "dietary_habits": "Cukup",
    "suicidal_thoughts": "Pernah",
    "study_hours": 8,
    "financial_stress": 3,
    "family_history": "Tidak"
}

result = predict_depression(test_previous)
print("=" * 60)
print("TEST 4: YOUR PREVIOUS MOBILE APP TEST")
print("=" * 60)
print(f"Prediction: {result['prediction_label']}")
print(f"Confidence: {result['confidence']}")
print(f"\nInput Features:")
for key, value in test_previous.items():
    print(f"  {key}: {value}")
print(f"\nNumeric Features: {result['features']}")

### Test Case 5: Borderline Case
**Mixed indicators to test model sensitivity**

In [ ]:
test_borderline = {
    "gender": "Laki-laki",
    "age": 23,
    "academic_pressure": 3,
    "study_satisfaction": 3,
    "sleep_duration": "5-6 jam",
    "dietary_habits": "Cukup",
    "suicidal_thoughts": "Tidak Pernah",
    "study_hours": 7,
    "financial_stress": 3,
    "family_history": "Tidak"
}

result = predict_depression(test_borderline)
print("=" * 60)
print("TEST 5: BORDERLINE/NEUTRAL CASE")
print("=" * 60)
print(f"Prediction: {result['prediction_label']}")
print(f"Confidence: {result['confidence']}")
print(f"\nInput Features:")
for key, value in test_borderline.items():
    print(f"  {key}: {value}")
print(f"\nNumeric Features: {result['features']}")

### Batch Test: Systematic Feature Analysis

In [ ]:
# Test multiple scenarios systematically
test_scenarios = [
    {
        "name": "Perfect Health",
        "data": {
            "gender": "Perempuan", "age": 20, "academic_pressure": 1,
            "study_satisfaction": 5, "sleep_duration": "Lebih dari 8 jam",
            "dietary_habits": "Sehat", "suicidal_thoughts": "Tidak Pernah",
            "study_hours": 5, "financial_stress": 1, "family_history": "Tidak"
        }
    },
    {
        "name": "High Academic Pressure Only",
        "data": {
            "gender": "Laki-laki", "age": 21, "academic_pressure": 5,
            "study_satisfaction": 4, "sleep_duration": "7-8 jam",
            "dietary_habits": "Sehat", "suicidal_thoughts": "Tidak Pernah",
            "study_hours": 9, "financial_stress": 2, "family_history": "Tidak"
        }
    },
    {
        "name": "Poor Sleep Only",
        "data": {
            "gender": "Perempuan", "age": 22, "academic_pressure": 2,
            "study_satisfaction": 4, "sleep_duration": "Kurang dari 5 jam",
            "dietary_habits": "Sehat", "suicidal_thoughts": "Tidak Pernah",
            "study_hours": 6, "financial_stress": 2, "family_history": "Tidak"
        }
    },
    {
        "name": "Multiple Risk Factors",
        "data": {
            "gender": "Laki-laki", "age": 24, "academic_pressure": 5,
            "study_satisfaction": 1, "sleep_duration": "Kurang dari 5 jam",
            "dietary_habits": "Tidak Sehat", "suicidal_thoughts": "Tidak Pernah",
            "study_hours": 12, "financial_stress": 5, "family_history": "Ya"
        }
    },
    {
        "name": "Only Suicidal Thoughts",
        "data": {
            "gender": "Perempuan", "age": 19, "academic_pressure": 1,
            "study_satisfaction": 5, "sleep_duration": "7-8 jam",
            "dietary_habits": "Sehat", "suicidal_thoughts": "Pernah",
            "study_hours": 4, "financial_stress": 1, "family_history": "Tidak"
        }
    }
]

print("\n" + "=" * 70)
print("SYSTEMATIC BATCH TESTING")
print("=" * 70)

results_df = []

for scenario in test_scenarios:
    result = predict_depression(scenario['data'])
    
    results_df.append({
        'Scenario': scenario['name'],
        'Prediction': result['prediction_label'],
        'Confidence': result['confidence'],
        'Suicidal Thoughts': scenario['data']['suicidal_thoughts'],
        'Academic Pressure': scenario['data']['academic_pressure'],
        'Sleep': scenario['data']['sleep_duration']
    })

results_table = pd.DataFrame(results_df)
print("\n", results_table.to_string(index=False))

### Feature Importance Analysis (if model supports it)

In [ ]:
# Check if model has feature_importances_ attribute
if hasattr(model, 'feature_importances_'):
    feature_names = [
        'gender', 'age', 'academic_pressure', 'study_satisfaction',
        'sleep_duration', 'dietary_habits', 'suicidal_thoughts',
        'study_hours', 'financial_stress', 'family_history'
    ]
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n" + "=" * 60)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("=" * 60)
    print(importance_df.to_string(index=False))
    
    # Check if suicidal_thoughts is dominating
    suicidal_importance = importance_df[importance_df['Feature'] == 'suicidal_thoughts']['Importance'].values[0]
    print(f"\n⚠️ Suicidal Thoughts Importance: {suicidal_importance:.4f}")
    print(f"   {'WARNING: This feature may be dominating predictions!' if suicidal_importance > 0.5 else 'Feature weight seems reasonable'}")
else:
    print("\nℹ️ Model does not support feature_importances_ (might be SVM, Neural Network, etc.)")
    print("   We can still analyze predictions but not feature weights directly.")

### Diagnosis: Is the Model Overfitting on Suicidal Thoughts?

In [ ]:
print("\n" + "=" * 70)
print("OVERFITTING DIAGNOSIS")
print("=" * 70)

# Test 1: Low risk + suicidal thoughts
test1 = predict_depression({
    "gender": "Perempuan", "age": 19, "academic_pressure": 1,
    "study_satisfaction": 5, "sleep_duration": "7-8 jam",
    "dietary_habits": "Sehat", "suicidal_thoughts": "Pernah",
    "study_hours": 4, "financial_stress": 1, "family_history": "Tidak"
})

# Test 2: High risk + no suicidal thoughts
test2 = predict_depression({
    "gender": "Laki-laki", "age": 24, "academic_pressure": 5,
    "study_satisfaction": 1, "sleep_duration": "Kurang dari 5 jam",
    "dietary_habits": "Tidak Sehat", "suicidal_thoughts": "Tidak Pernah",
    "study_hours": 12, "financial_stress": 5, "family_history": "Ya"
})

print(f"\nTest A - Low Risk + Suicidal Thoughts:")
print(f"  Prediction: {test1['prediction_label']} (Confidence: {test1['confidence']})")

print(f"\nTest B - High Risk + No Suicidal Thoughts:")
print(f"  Prediction: {test2['prediction_label']} (Confidence: {test2['confidence']})")

print("\n" + "-" * 70)
print("CONCLUSION:")
if test1['prediction'] == 1 and test2['prediction'] == 0:
    print("⚠️  OVERFITTING DETECTED!")
    print("   The model predicts depression for low-risk + suicidal thoughts,")
    print("   but NOT for high-risk + no suicidal thoughts.")
    print("   This suggests the model is over-relying on suicidal_thoughts feature.")
elif test1['prediction'] == 1 and test2['prediction'] == 1:
    print("✅ Model seems balanced.")
    print("   Both scenarios are correctly identified as high risk.")
else:
    print("ℹ️  Mixed results - manual review recommended.")
    print("   Compare the confidence scores to understand model behavior.")
print("=" * 70)